# Part 2: Sparse Attention Vectors (SAVs) on CC-OCR doc/table parsing (Kaggle, T4)

**Read `PROJECT_CONTEXT.md` first.** Kaggle version of `Part2_SAVs_Colab.ipynb`: same code, tasks, prompts and
settings, so results are comparable with the Colab runs. Only `MODEL_KEY` changes.

| MODEL_KEY | Model | Owner |
|---|---|---|
| `qwen25vl_7b` | Qwen/Qwen2.5-VL-7B-Instruct (4-bit) | Ujjwal |
| `phi4_mm` | microsoft/Phi-4-multimodal-instruct (fp16) | Ujjwal |
| `llama32v_11b` | meta-llama/Llama-3.2-11B-Vision-Instruct (4-bit, gated) | Anvesh |
| `gemma3_12b` | google/gemma-3-12b-it (4-bit, gated) | Pratham |

**One-time Kaggle setup** (right-hand panel of the notebook editor):
1. **Settings → Accelerator → GPU T4 x2.** Not P100: it has no bfloat16, which Gemma-3 needs.
2. **Settings → Internet → On.** Needs a phone-verified Kaggle account. Used for pip, Hugging Face and git.
3. **Settings → Persistence → Files only.** `/kaggle/working`, where results are written, then survives between
   interactive sessions, so *Run All* again continues where it stopped.
4. **Add-ons → Secrets:** add `HF_TOKEN` (a Hugging Face *read* token) and tick it for this notebook.
   Required for the gated models (Llama, Gemma); accept the license on the model's HF page first.
5. *Optional:* attach a Kaggle dataset that contains `doc/doc_scan_eng_75.tsv` etc. (the CC-OCR
   `doc_parsing` folder). Without one, the 4 TSVs (~68 MB) are downloaded from Hugging Face.

**Long runs.** Kaggle gives 30 GPU hours/week and stops a session after 12 h. For Part 2B use
**Save Version → Save & Run All (Commit)**, which runs in the background. Each run stops cleanly after
`TIME_BUDGET_HOURS` so its output is saved. To continue: **Add Input → Your Work → this notebook's output**
and commit again. Cell 2 copies the finished images from `/kaggle/input` back in and skips them.

**First time?** Set `SMOKE_TEST = True` and run all (~15 min). It runs every phase on a few images, with short
generations, into a separate `SAVs_smoke` folder. If it finishes without errors, set it back to `False`.

In [ ]:
# ===== 1. CONFIG: the only cell you should edit =====
MODEL_KEY = "gemma3_12b"           # qwen25vl_7b | phi4_mm | llama32v_11b | gemma3_12b
RUN_PART_2A = True                 # SAV extraction + zero-shot (~1.5-3 h per model)
RUN_PART_2B = True                 # full parsing under 3 configs (slow; resumable)
N_PARSE_PER_SPLIT = 25             # images per split for Part 2B (25 x 4 splits x 3 configs = 300 generations)
DTYPE_OVERRIDE = None              # e.g. "float32" if you get NaN / FloatingPointError
SMOKE_TEST = False                 # True = quick end-to-end check on a few images (results go to SAVs_smoke)
TIME_BUDGET_HOURS = 10.5           # stop starting new work after this, so a 12 h Kaggle session ends cleanly
SINGLE_GPU = True                  # "T4 x2": keep the model on one T4 (it fits; avoids cross-GPU overhead)
REPO_URL = "https://github.com/UjjWaL-0911/Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing.git"

In [ ]:
# ===== 2. Get the code, find the data, restore earlier results =====
import glob, os, shutil, subprocess, sys, time
T_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working")
os.environ.setdefault("USE_TF", "0")                 # Kaggle's image ships TensorFlow; keep transformers off it
if SINGLE_GPU:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # must happen before torch is imported

if ON_KAGGLE:
    # Code lives outside /kaggle/working so it is not saved as output; results do.
    # Sparse checkout = top-level .py files + savs/ only (skips ~480 MB of Part 1 HTML reports).
    REPO_DIR = "/tmp/btp_code"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "add", "savs"], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    RESULTS_ROOT = "/kaggle/working/SAVs_smoke" if SMOKE_TEST else "/kaggle/working/SAVs"
else:  # running outside Kaggle (e.g. a local smoke test): use the repo this notebook is in
    REPO_DIR = os.getcwd()
    RESULTS_ROOT = os.path.join("Evaluation_Results", "SAVs_smoke" if SMOKE_TEST else "SAVs")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Data: an attached Kaggle dataset with the doc_parsing TSVs, else dataset.py downloads them from Hugging Face.
hits = glob.glob("/kaggle/input/**/doc/doc_scan_eng_75.tsv", recursive=True)
if hits and "CC_OCR_DIR" not in os.environ:
    os.environ["CC_OCR_DIR"] = os.path.dirname(os.path.dirname(hits[0]))
    print("Using CC-OCR data from", os.environ["CC_OCR_DIR"])

def restore_results(dst_model_dir):
    """Copy finished results of an earlier run (attached via Add Input) into RESULTS_ROOT; never overwrite newer."""
    srcs =[d for d in glob.glob(f"/kaggle/input/**/{MODEL_KEY}", recursive=True)
            if os.path.isdir(d) and (os.path.isdir(os.path.join(d, "activations")) or os.path.isdir(os.path.join(d, "parsing")))
            and (os.path.basename(os.path.dirname(d)) == "SAVs_smoke") == SMOKE_TEST]
    for src in srcs:
        n = 0
        for f in glob.glob(os.path.join(src, "**", "*"), recursive=True):
            if not os.path.isfile(f):
                continue
            dst = os.path.join(dst_model_dir, os.path.relpath(f, src))
            if os.path.exists(dst) and not (f.endswith(".jsonl") and
                                            sum(1 for _ in open(f)) > sum(1 for _ in open(dst))):
                continue
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(f, dst)
            n += 1
        print(f"Restored {n} result files from {src}")

if ON_KAGGLE:
    restore_results(os.path.join(RESULTS_ROOT, MODEL_KEY))
print("Results folder:", os.path.abspath(RESULTS_ROOT))

In [ ]:
# ===== 3. Install dependencies (Phi-4 needs old pinned versions) =====
if ON_KAGGLE:
    if MODEL_KEY == "phi4_mm":
        %pip install -q "transformers==4.48.2" "peft==0.13.2" "accelerate==1.3.0" soundfile scipy backoff
    else:
        %pip install -q -U "transformers>=4.50" accelerate bitsandbytes
    %pip install -q rapidfuzz apted distance lxml tabulate nltk
import transformers, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| GPU:", gpu or "NONE - Settings -> Accelerator -> GPU T4 x2!")
if gpu and "P100" in gpu:
    print("WARNING: P100 has no bfloat16 and is not what the team used. Switch the accelerator to GPU T4 x2.")

**Gated models (Llama-3.2-Vision, Gemma-3):** first accept the license on the model's Hugging Face page with your
account, create a *read* token at huggingface.co/settings/tokens, and add it under **Add-ons → Secrets** as
`HF_TOKEN` (tick it for this notebook). Qwen and Phi-4 do not need this but it still speeds up downloads.

In [ ]:
# ===== 4. Hugging Face login =====
from huggingface_hub import login
token = os.environ.get("HF_TOKEN")
if not token and ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("No HF_TOKEN secret found (required for llama32v_11b and gemma3_12b):", e)
if token:
    login(token)
    print("Logged in to Hugging Face.")

In [ ]:
# ===== 5. Load model + attention-head hooks, sanity check on one image =====
from savs.models import load_model, MODELS
from savs.extract import HeadExtractor
from savs.pipeline import run_extraction, run_parsing, write_run_info, progress
from savs.tasks import TASKS, PARSE_CONFIGS, EXTRACT_MAX_EDGE, PARSE_MAX_NEW_TOKENS, task_prompt
from savs.preprocess import preprocess
from dataset import load_split, decode_image, SPLITS

# Smoke test: every phase, but on a few images and with short generations (8 + 5 images/split is the minimum
# that still gives Part 2B's routing analysis its 20 images).
N_EXTRACT = 8 if SMOKE_TEST else None
N_PARSE = 5 if SMOKE_TEST else N_PARSE_PER_SPLIT
PARSE_TOKENS = 64 if SMOKE_TEST else PARSE_MAX_NEW_TOKENS
DEADLINE = T_START + TIME_BUDGET_HOURS * 3600

def out_of_time():
    if time.time() > DEADLINE:
        print(f"Time budget of {TIME_BUDGET_HOURS} h used up: stopping here. Commit/run again to continue.")
        return True
    return False

t0 = time.time()
vlm = load_model(MODEL_KEY, dtype=DTYPE_OVERRIDE)
ext = HeadExtractor(vlm)
print(f"Loaded {MODELS[MODEL_KEY]['hf_id']} in {time.time()-t0:.0f}s | {ext.describe()}")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(write_run_info(vlm, ext, root=RESULTS_ROOT))

row = load_split("Photoed_Docs", 1).iloc[0]
img = preprocess(decode_image(row["image"]), max_edge=EXTRACT_MAX_EDGE)
ext.reset(); t0 = time.time()
ans = vlm.run(img, task_prompt("acquisition", row), max_new_tokens=8)
acts = ext.collect()
print(f"Zero-shot answer: {ans!r} (truth: photographed) | activations {tuple(acts.shape)} | {time.time()-t0:.1f}s")

In [ ]:
# ===== 6. PART 2A: SAV activations + zero-shot answers for all classification tasks (+ 'parse') =====
if RUN_PART_2A:
    for task in TASKS:
        if out_of_time():
            break
        run_extraction(vlm, ext, task, n_per_split=N_EXTRACT, root=RESULTS_ROOT)
progress(MODEL_KEY, root=RESULTS_ROOT, n_parse_per_split=N_PARSE)

In [ ]:
# ===== 7. Quick look at YOUR model's Part 2A results (CPU, ~1 min) =====
from savs.analysis import analyze, to_markdown
from IPython.display import Markdown, display
display(Markdown(to_markdown(analyze([MODEL_KEY], root=RESULTS_ROOT))))

In [ ]:
# ===== 8. PART 2B: full parsing under each pre-processing config (slow, resumable) =====
# Rough cost on a T4: 1-3 min per image. Watch the first few timings and lower N_PARSE_PER_SPLIT if needed
# (but tell the team: everyone should use the same N). Runs one split at a time so the time budget is checked often.
if RUN_PART_2B:
    for cfg in PARSE_CONFIGS:
        for split in SPLITS:
            if out_of_time():
                break
            run_parsing(vlm, cfg, n_per_split=N_PARSE, max_new_tokens=PARSE_TOKENS, root=RESULTS_ROOT, splits=(split,))
progress(MODEL_KEY, root=RESULTS_ROOT, n_parse_per_split=N_PARSE)
report_md = to_markdown(analyze([MODEL_KEY], root=RESULTS_ROOT))
with open(os.path.join(RESULTS_ROOT, f"summary_{MODEL_KEY}.md"), "w", encoding="utf-8") as f:
    f.write(report_md + "\n")
display(Markdown(report_md))

In [ ]:
# ===== 9. Hand-off: zip your results so Ujjwal can combine all models =====
# Same layout as the Colab zip (<MODEL_KEY>/activations, <MODEL_KEY>/parsing, run_info.json).
zip_dir = "/kaggle/working" if ON_KAGGLE else RESULTS_ROOT
zip_path = shutil.make_archive(os.path.join(zip_dir, f"SAVs_results_{MODEL_KEY}" + ("_smoke" if SMOKE_TEST else "")),
                               "zip", root_dir=RESULTS_ROOT, base_dir=MODEL_KEY)
print("Download from the notebook's Output tab and share with Ujjwal:", zip_path)